# Práctica Día 02 — Tu primer MLP sobre Telco Customer Churn

Asignatura: Deep Learning para Business Analytics — Comillas (ICADE).
Profesor: Eduardo C. Garrido-Merchán · `ecgarrido@comillas.edu`

Objetivo: entrenar tu primer MLP en PyTorch sobre Telco Customer Churn y compararlo con una regresión logística.
Dataset: <https://www.kaggle.com/datasets/blastchar/telco-customer-churn>. Si no lo tienes, se usa un sintético.

## 1. Datos

El sintetico NO es un modelo lineal en los log-odds. Contiene cuatro estructuras que una
logistica sobre las features crudas no puede representar: un riesgo de baja **no monotono**
en `tenure` (pico de onboarding, picos de renovacion en el mes 12 y el 24), un **umbral**
sobre el precio por servicio `monthly_charges/(1+extra_lines)`, y dos **interacciones con
cambio de signo** (fibra x llamadas a soporte, promocion x tipo de contrato). Por eso aqui
el MLP gana. Con el CSV real de Kaggle veras que la diferencia casi desaparece: en ese
dataset la senal es practicamente lineal y la logistica es un baseline durisimo. Las dos
lecciones cuentan.

In [ ]:
import sys; sys.path.insert(0, '.')
from practica_02 import load_data, prepare
df = load_data(None)
df.head()

## 2. Logística baseline

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score, f1_score
X, y = prepare(df)
# Tres splits: train (ajuste), val (elegir la epoca del MLP), test (medir).
# Elegir la epoca con el test seria mirar la respuesta del examen.
X_fit, X_te, y_fit, y_te = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_tr, X_va, y_tr, y_va = train_test_split(X_fit, y_fit, test_size=0.2, random_state=0, stratify=y_fit)
sc = StandardScaler().fit(X_tr)
X_tr_s, X_va_s, X_te_s = sc.transform(X_tr), sc.transform(X_va), sc.transform(X_te)
# La logistica se ajusta sobre train+val: mismo numero de ejemplos vistos, pelea justa.
sc_lr = StandardScaler().fit(X_fit)
lr = LogisticRegression(max_iter=2000).fit(sc_lr.transform(X_fit), y_fit)
p_lr = lr.predict_proba(sc_lr.transform(X_te))[:, 1]
{'auc': roc_auc_score(y_te, p_lr),
 'acc': accuracy_score(y_te, (p_lr>=0.5).astype(int)),
 'recall': recall_score(y_te, (p_lr>=0.5).astype(int)),
 'f1': f1_score(y_te, (p_lr>=0.5).astype(int))}

## 3. MLP con PyTorch

In [ ]:
from practica_02 import train, metrics
m, hist = train(X_tr_s, y_tr, X_va_s, y_va, lr=1e-3, epochs=120)
print('mejor epoca:', hist['best_epoch'], '| perdida val:', round(hist['best_val'], 4))
metrics(m, X_te_s, y_te)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(hist['train'], label='train'); plt.plot(hist['val'], label='val')
plt.xlabel('época'); plt.ylabel('pérdida'); plt.legend(); plt.show()

## 4. Ejemplos a mano (sin librerías)

**Ejercicio 4.1.** Toma el primer cliente del test, sus features y los pesos de la capa 1 del MLP. Calcula a mano (con numpy básico) la salida de la primera neurona. ¿Coincide con lo que da PyTorch?

**Ejercicio 4.2.** Cambia el learning rate a $10^{-1}$, vuelve a entrenar y comenta qué pasa con la pérdida en train y validación.

## 5. Análisis (sin IA)

En 8 líneas, responde:

1. ¿Qué métrica es más relevante en un caso de retención de clientes y por qué?
2. Si retener cuesta 30 € por cliente y un cliente perdido vale 500 €, ¿qué umbral elegirías? Justifica con expected value.
3. Defiende delante del director técnico si el MLP merece la pena sobre la logística.

## 6. Declaración de uso de IA

| Sección | Herramienta | Para qué | Edición posterior |
|---|---|---|---|
| 1–3 | … | … | … |
| 4–5 | Ninguna | — | — |